In [26]:
import numpy as np
from scipy.optimize import minimize

# load params
mu = np.load("project3_outputs/mu_hat.npy")
Sigma = np.load("project3_outputs/Sigma_hat.npy")
gamma = 3.0
ones = np.array([1, 1, 1])
assets = ["SPY", "IEF", "GLD"]

print(f"mu    = {mu}")
print(f"gamma = {gamma}")
print(f"Sigma =")
for row in Sigma:
    print(f"        {row}")
print()

mu    = [ 0.00051575 -0.00024641  0.00032329]
gamma = 3.0
Sigma =
        [1.08500755e-04 5.42930040e-06 1.54645268e-05]
        [5.4293004e-06 2.6382581e-05 1.8884914e-05]
        [1.54645268e-05 1.88849140e-05 8.14528009e-05]



Unconstrained Optimal Portfolio
Closed form: w* = Σ⁻¹1/(1ᵀΣ⁻¹1) + (1/γ) Σ⁻¹(μ - (1ᵀΣ⁻¹μ)/(1ᵀΣ⁻¹1) · 1)

In [27]:
#Σ⁻¹
sigma_inv = np.linalg.inv(Sigma)

#Σ⁻¹𝟏
sigma_inv_ones = sigma_inv @ ones


GMVP = sigma_inv_ones / (ones @ sigma_inv_ones)

speculative = 1 / gamma * sigma_inv @ (mu - (ones @ sigma_inv @ mu) / (ones @ sigma_inv_ones) * ones)

w_star = GMVP + speculative

# stats
portfolio_mean = mu @ w_star
portfolio_var = w_star @ Sigma @ w_star
portfolio_utility = portfolio_mean - 0.5 * gamma * portfolio_var

print(f"GMVP component:        {np.round(GMVP, 6)}")
print(f"Speculative component: {np.round(speculative, 6)}")

for i, asset in enumerate(assets):
    print(f"w*_{asset} = {w_star[i]:+.6f}")

print(f"\nSum of weights:     {w_star.sum():.6f}")
print(f"Portfolio mean:     {portfolio_mean:.8f}")
print(f"Portfolio variance: {portfolio_var:.10f}")
print(f"Portfolio utility:  {portfolio_utility:.8f}")
print()

GMVP component:        [0.159458 0.773435 0.067107]
Speculative component: [ 1.726325 -4.004652  2.278327]
w*_SPY = +1.885783
w*_IEF = -3.231217
w*_GLD = +2.345434

Sum of weights:     1.000000
Portfolio mean:     0.00252705
Portfolio variance: 0.0008937694
Portfolio utility:  0.00118639



LONG-ONLY OPTIMAL PORTFOLIO: same objective, except all weights must be >= 0

In [28]:
# negative utility formula since scipy can only minimize. (Objective function)
def neg_utility(w):
    return -(mu @ w - 0.5 * gamma * (w @ Sigma @ w))

# weights sum must be == 1
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
bounds = [(0, None), (0, None), (0, None)]

best_util = -np.inf
best_w = None

for guess in [[1/3, 1/3, 1/3], [0.6, 0.1, 0.3], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7], [1.0, 0.0, 0.0]]:
    res = minimize(neg_utility, x0=guess, method="SLSQP", bounds=bounds, constraints=constraints, options={"ftol": 1e-15, "maxiter": 1000})
    if res.success and -res.fun > best_util:
        # update best score
        best_util = -res.fun
        best_w = res.x

w_long = best_w

portfolio_mean_long = mu @ w_long
portfolio_var_long = w_long @ Sigma @ w_long
portfolio_utility_long = portfolio_mean_long - 0.5 * gamma * portfolio_var_long

for i, name in enumerate(assets):
    print(f"w*_{name} =            {w_long[i]:+.6f}")
print(f"\nSum of weights:     {w_long.sum():.6f}")
print(f"Portfolio mean:     {portfolio_mean_long:.8f}")
print(f"Portfolio variance: {portfolio_var_long:.10f}")
print(f"Portfolio utility:  {portfolio_utility_long:.8f}")

w*_SPY =            +0.818372
w*_IEF =            +0.000000
w*_GLD =            +0.181628

Sum of weights:     1.000000
Portfolio mean:     0.00048079
Portfolio variance: 0.0000799508
Portfolio utility:  0.00036087


In [29]:
print("Verification:")

# how much utility improves with a bit more weight
grad = mu - gamma * (Sigma @ w_long)

# marginal utility of active assets
l = np.mean(grad[w_long > 1e-8])  

for i, asset in enumerate(assets):
    if w_long[i] > 1e-8:
        nu_i = 0.0
        print(f"{asset}: w={w_long[i]:.6f} > 0, nu=0 (active, marginal util={grad[i]:.8f})")
    else:
        nu_i = l - grad[i]
        print(f"{asset}: w=0, nu={nu_i:.8f} > 0 (excluded)")

Verification:
SPY: w=0.818372 > 0, nu=0 (active, marginal util=0.00024094)
IEF: w=0, nu=0.00051097 > 0 (excluded)
GLD: w=0.181628 > 0, nu=0 (active, marginal util=0.00024094)


In [30]:
# Comparison between unconstrained and long-only
utility_loss = portfolio_utility - portfolio_utility_long
dist = np.linalg.norm(w_star - w_long)

print(f"Euclidean distance: {dist:.6f}")
print(f"Utility loss:       {utility_loss:.10f}\n")

for i, asset in enumerate(assets):
    print(f"{asset}:  unconstrained={w_star[i]:+.6f}   long only={w_long[i]:+.6f}")
print()
for i, asset in enumerate(assets):
    if w_star[i] < -1e-6:
        print(f"{asset} was shorted in the unconstrained solution but zeroed out by long-only")

Euclidean distance: 4.032640
Utility loss:       0.0008255263

SPY:  unconstrained=+1.885783   long only=+0.818372
IEF:  unconstrained=-3.231217   long only=+0.000000
GLD:  unconstrained=+2.345434   long only=+0.181628

IEF was shorted in the unconstrained solution but zeroed out by long-only
